<a href="https://colab.research.google.com/github/Bhuvana-Manogar/Constrained-Object-Detection-Reasoning-API/blob/main/notebooks/train_on_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RT-DETR PPE fine-tuning -- Colab runner

**Run every cell below, IN ORDER, top to bottom. Don't skip any.**

Before starting: Runtime menu (top left) -> Change runtime type -> Hardware accelerator: T4 GPU -> Save.

You will be prompted partway through to upload a file called `data_raw.zip` --
prepare it BEFORE you start by running this on your own Windows machine:
```
cd C:\Users\admin\Downloads\RAP\ppe-detection-api
Compress-Archive -Path data\raw -DestinationPath data_raw.zip
```
That creates `data_raw.zip` inside your `ppe-detection-api` folder -- have it ready to pick when Colab asks.

## Cell 1 -- Install dependencies and confirm GPU

In [1]:
!pip install -q ultralytics
!nvidia-smi  # record this output -- goes in your memo's reproducibility section

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.0 MB/s eta 0:00:00
Wed Sep  9 04:14:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0

## Cell 2 -- Clone your GitHub repo (gets you src/, api/, data/data.yaml)

In [2]:
!git clone https://github.com/Bhuvana-Manogar/Constrained-Object-Detection-Reasoning-API.git repo
%cd repo

Cloning into 'repo'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 21 (delta 0), reused 21 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 17.88 KiB | 17.88 MiB/s, done.
/content/repo


## Cell 3 -- Upload your dataset zip
Running this cell opens a file-picker button. Click it and select `data_raw.zip` from your computer.

In [3]:
from google.colab import files
uploaded = files.upload()

Saving data_raw.zip to data_raw.zip


## Cell 4 -- Unzip the dataset into the right place

In [4]:
!mkdir -p data/raw
!unzip -q data_raw.zip -d data/raw_tmp
# Compress-Archive on Windows preserves the 'raw' folder itself inside the zip,
# so the real content lands one level deeper -- this moves it up to data/raw.
!mv data/raw_tmp/raw/* data/raw/ 2>/dev/null || mv data/raw_tmp/* data/raw/
!ls data/raw

images	labels


## Cell 5 -- Verify the dataset landed correctly
You should see two numbers, both around 2801 (or whatever your local count showed).

In [5]:
import os
print('images:', len(os.listdir('data/raw/images')))
print('labels:', len(os.listdir('data/raw/labels')))

images: 2801
labels: 2801


## Cell 6 -- Split (same seed as your local run, so results match)

In [6]:
!python src/split_dataset.py --source data/raw --dest data/split --seed 42

Split complete.
  train: 2101 images
  val: 422 images
  test: 278 images
Seed used: 42 (record this in your memo for reproducibility)


## Cell 7 -- Train (this is the slow one -- can take 30-90+ minutes depending on epochs)

In [7]:
!python src/train.py --data data/data.yaml --epochs 100 --imgsz 640 --batch 16 --device 0

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.144 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1

## Cell 8 -- Evaluate on the held-out test split

In [11]:
!find runs -name "best.pt"

runs/detect/runs/ppe_rtdetr/weights/best.pt


In [12]:
!python src/evaluate.py --weights runs/detect/runs/ppe_rtdetr/weights/best.pt --data data/data.yaml --split test

Ultralytics 8.4.144 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary (fused): 315 layers, 32,004,290 parameters, 0 gradients, 105.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 25.8±11.3 MB/s, size: 55.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/repo/data/split/test/labels... 278 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 278/278 623.0it/s 0.4s
val: New cache created: /content/repo/data/split/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 1.3it/s 13.3s
                   all        278       3809      0.918      0.865      0.905      0.662
               Hardhat        119        311      0.906      0.803       0.86      0.588
                  Mask        107        161      0.932      0.919      0.939       0.

## Cell 9 -- Download your trained weights back to your computer
This downloads `best.pt` -- move it into your local repo's `models/best.pt` afterward.

In [14]:
from google.colab import files
files.download('runs/detect/runs/ppe_rtdetr/weights/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 10 -- Also download the confusion matrix + PR curve images for your memo's failure-case section

In [15]:
import glob
from google.colab import files
for f in glob.glob('runs/detect/val/confusion_matrix.png') + glob.glob('runs/detect/val/*curve*.png'):
    print('downloading', f)
    files.download(f)

downloading runs/detect/val/confusion_matrix.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

downloading runs/detect/val/BoxP_curve.png


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

downloading runs/detect/val/BoxR_curve.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>